# Yer adı sözlüğü (v1.1): geliştirme setinde yeniden değerlendirme

GPU **gerekmez** (yalnız 3 modelle tahmin yapılır, CPU'da birkaç dakika). Final defteri bittikten sonra çalıştır.
Mühürlü test çalıştırılmaz: sözlük tasarlanırken mühürlü cümleler görüldüğü için v1.1'in mühürlü sonucu temiz sayılmaz.
**Tümünü çalıştır** → son hücrenin çıktısını yapıştır.


In [ ]:
import os, pathlib, shutil, subprocess, zipfile, json, torch
print("GPU:", torch.cuda.is_available(), "(gerekmez; yalnız tahmin yapılır)")
from google.colab import drive
drive.mount('/content/drive')
# Klasörler başka bir hesaptan paylaşılıp kısayol olarak eklenmiş olabilir: kısayol hedeflerinde de ara
ROOTS = [pathlib.Path('/content/drive/MyDrive'), pathlib.Path('/content/drive/.shortcut-targets-by-id')]
def walk_dirs():
    for r in ROOTS:
        if r.exists():
            for d, subdirs, files in os.walk(r, followlinks=True):
                yield pathlib.Path(d), subdirs, files
DRIVE = pathlib.Path('/content/drive/MyDrive/turkce-llm-dlp')
if not DRIVE.exists():
    DRIVE = next((d / 'turkce-llm-dlp' for d, sub, _ in walk_dirs() if 'turkce-llm-dlp' in sub), DRIVE)
assert (DRIVE / 'runs').exists(), f'{DRIVE}/runs bulunamadı: turkce-llm-dlp klasörünü bu hesapta Drive\'ım\'a kısayol olarak ekle'
print('Drive klasörü:', DRIVE)
os.chdir('/content')
shutil.rmtree('/content/proje', ignore_errors=True)
subprocess.run(['git', 'clone', '--depth', '1', 'https://github.com/irmakyaprakk/turkce-llm-dlp.git', '/content/proje'], check=True)
bundles = set()
for d, _, files in walk_dirs():
    if 'BUNDLE_INFO.txt' in files: bundles.add(d)
    bundles |= {d / f for f in files if f.startswith('turkce-dlp-colab-bundle') and f.endswith('.zip')}
for src in sorted(bundles, key=lambda p: p.name):
    tmp = pathlib.Path('/content/_paket'); shutil.rmtree(tmp, ignore_errors=True)
    shutil.copytree(src, tmp) if src.is_dir() else zipfile.ZipFile(src).extractall(tmp)
    root = next((p.parent for p in tmp.rglob('BUNDLE_INFO.txt')), tmp)
    if (root / 'data').exists():
        shutil.copytree(root / 'data', '/content/proje/data', dirs_exist_ok=True)
os.chdir('/content/proje')
assert pathlib.Path('data/ner/processed/augmented-v3/train.jsonl').exists(), 'augmented-v3 verisi yok (v5 paketi Drive\'da olmalı)'
!pip -q install -r training/ner/requirements.txt
print(subprocess.run(['git', 'log', '--oneline', '-1'], capture_output=True, text=True).stdout)

In [ ]:
# Geliştirme seti (realistic-v2): üç seed, sözlüklü ve sözlüksüz hibrit karşılaştırması
# Mühürlü test burada ÇALIŞTIRILMAZ (bkz. docs/evaluation-protocol.md).
!python training/ner/build_gold_set.py --out-dir data/ner/processed/realistic-v2 > /dev/null
files = []
for seed in (42, 43, 44):
    model = DRIVE / 'runs' / f'augmented_v3_seed{seed}' / 'model'
    preds = f'/content/gelistirme-{seed}.jsonl'
    open(preds, 'w').close()
    for split in ('dev', 'test'):
        !USE_TF=0 python training/ner/evaluate_model.py --model-dir "{model}" --data-dir data/ner/processed/realistic-v2 --split {split} --dump-predictions /content/p.jsonl > /dev/null
        !cat /content/p.jsonl >> {preds}
    shutil.copy(preds, DRIVE / 'runs' / f'augmented_v3_seed{seed}' / 'gelistirme-pred.jsonl')
    target = f'/content/gelistirme-v11-{seed}.json'
    !python training/ner/evaluate_rules.py --data-dir data/ner/processed/realistic-v2 --splits dev test --with-predictions {preds} --output-json {target} > /dev/null
    files.append(target)
out = subprocess.run(['python', 'training/ner/aggregate_seeds.py', *files], capture_output=True, text=True, check=True).stdout
(DRIVE / 'runs' / 'gelistirme-v11-summary.json').write_text(out)
metrics = json.loads(out)['metrics']
rows = {}
for variant, label in (('pipeline_model_plus_rules_no_gazetteer', 'v1 (sözlüksüz)'), ('pipeline_model_plus_rules', 'v1.1 (sözlüklü)')):
    rows[label] = {k.split('.')[-1]: f"{v['mean']:.3f} ± {v['std']:.3f}" for k, v in metrics.items()
                   if k.startswith(variant + '.') and any(x in k for x in ('overall_f1', 'overall_precision', 'overall_recall', 'location_f1', 'location_precision', 'location_recall', 'overall_coverage', 'location_coverage'))}
print(json.dumps(rows, ensure_ascii=False, indent=1))
